# Checkpoint C1: Chunker.py — Tach tai lieu thanh chunks

Day la ket qua chay chunker.py. Notebook nay load cac tai lieu chinh sach HR,
tach thanh chunks theo heading, va hien thi ket qua.

**Cach su dung:** Chay tat ca cells de xem chunker hoat dong.

In [ ]:
import sys
import json
import os
from pathlib import Path

# Them scripts path de import chunker module
SCRIPTS_DIR = Path("../../templates/skills/hr-policy-qa-skill/scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print(f"Scripts dir: {SCRIPTS_DIR}")
print(f"Exists: {SCRIPTS_DIR.exists()}")
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
# Thu import chunker module
try:
    from chunker import (
        load_policy_files,
        chunk_markdown_by_heading,
        generate_metadata,
        run_pipeline,
    )
    HAS_CHUNKER = True
    print("Da import chunker module thanh cong!")
    print("  - load_policy_files")
    print("  - chunk_markdown_by_heading")
    print("  - generate_metadata")
    print("  - run_pipeline")
except ImportError as e:
    HAS_CHUNKER = False
    print(f"Khong the import chunker module: {e}")
    print("Se dinh nghia cac ham inline...")
    
    # Inline dinh nghia cac ham chunker (fallback)
    import re
    import uuid
    from typing import Any

    def _parse_frontmatter(text):
        match = re.match(r"^---\s*\n(.*?)\n---\s*\n", text, re.DOTALL)
        if not match:
            return {}, text
        frontmatter_str = match.group(1)
        body = text[match.end():]
        metadata = {}
        for line in frontmatter_str.strip().splitlines():
            if ":" in line:
                key, _, value = line.partition(":")
                metadata[key.strip()] = value.strip().strip('"')
        return metadata, body

    def load_policy_files(kb_dir):
        kb_path = Path(kb_dir)
        if not kb_path.exists():
            return []
        documents = []
        for md_file in sorted(kb_path.glob("*.md")):
            raw = md_file.read_text(encoding="utf-8")
            metadata, body = _parse_frontmatter(raw)
            doc_id = metadata.get("doc_id", md_file.stem)
            documents.append({
                "filename": md_file.name,
                "doc_id": doc_id,
                "metadata": metadata,
                "content": body,
            })
        return documents

    def chunk_markdown_by_heading(content, max_words=500, overlap_words=50):
        parts = re.split(r"(?=^##\s+)", content, flags=re.MULTILINE)
        chunks = []
        chunk_index = 0
        for part in parts:
            part = part.strip()
            if not part:
                continue
            heading_match = re.match(r"^##\s+(.+?)(?:\n|$)", part)
            if heading_match:
                section_name = heading_match.group(1).strip()
                body = part[heading_match.end():].strip()
            else:
                section_name = "\u1ed5ng quan"
                body = part.strip()
            if not body:
                continue
            words = body.split()
            if len(words) <= max_words:
                chunks.append({"section": section_name, "content": body, "chunk_index": chunk_index})
                chunk_index += 1
            else:
                start = 0
                while start < len(words):
                    end = start + max_words
                    sub_text = " ".join(words[start:end])
                    chunks.append({"section": section_name, "content": sub_text, "chunk_index": chunk_index})
                    chunk_index += 1
                    start += max_words - overlap_words
        return chunks

    print("Da dinh nghia inline functions thanh cong!")

In [ ]:
# Load 4 HR policy files
KB_DIR = Path("../../synthetic-data/hr-policies")

documents = load_policy_files(str(KB_DIR))

print(f"Da load {len(documents)} tai lieu chinh sach HR:")
print("=" * 60)
for doc in documents:
    doc_id = doc.get("doc_id", "?")
    filename = doc.get("filename", "?")
    content_len = len(doc.get("content", ""))
    word_count = len(doc.get("content", "").split())
    version = doc.get("metadata", {}).get("version", "?")
    print(f"  {doc_id} ({version}) — {filename}")
    print(f"    {content_len} ky tu, {word_count} tu")
    print()

if not documents:
    print("[CANH BAO] Khong tim thay tai lieu. Kiem tra duong dan:")
    print(f"  {KB_DIR.resolve()}")
    print(f"  Exists: {KB_DIR.exists()}")

In [ ]:
# Chay chunk_markdown_by_heading() tren policy-leave.md
leave_doc = None
for doc in documents:
    if "leave" in doc.get("filename", "").lower():
        leave_doc = doc
        break

if leave_doc:
    print(f"Chunking: {leave_doc['filename']} ({leave_doc['doc_id']})")
    print("=" * 60)
    
    chunks = chunk_markdown_by_heading(leave_doc["content"])
    
    print(f"Tong so chunks: {len(chunks)}\n")
    
    for i, chunk in enumerate(chunks):
        section = chunk["section"]
        content = chunk["content"]
        word_count = len(content.split())
        
        print(f"--- Chunk {i}: {section} ({word_count} tu) ---")
        # Hien thi 3 dong dau
        preview_lines = content.split("\n")[:3]
        for line in preview_lines:
            print(f"  {line[:80]}")
        if len(preview_lines) < content.count("\n") + 1:
            print(f"  ... (con them {content.count(chr(10)) - 2} dong)")
        print()
else:
    print("[LOI] Khong tim thay policy-leave.md trong danh sach tai lieu")

In [ ]:
# Chay chunking tren tat ca 4 file, hien thi bang tong hop
print("BANG TONG HOP CHUNKING")
print("=" * 65)
print(f"{'doc_id':<18} {'Filename':<22} {'# Chunks':<10} {'Avg words':<10}")
print("-" * 65)

all_chunks_all_docs = []

for doc in documents:
    doc_id = doc.get("doc_id", "?")
    filename = doc.get("filename", "?")
    chunks = chunk_markdown_by_heading(doc["content"])
    
    chunk_word_counts = [len(c["content"].split()) for c in chunks]
    avg_words = sum(chunk_word_counts) / len(chunk_word_counts) if chunk_word_counts else 0
    
    print(f"{doc_id:<18} {filename:<22} {len(chunks):<10} {avg_words:<10.0f}")
    
    for c in chunks:
        c["doc_id"] = doc_id
    all_chunks_all_docs.extend(chunks)

print("-" * 65)
print(f"{'TONG':<18} {'':<22} {len(all_chunks_all_docs):<10}")

In [ ]:
# Thu ChromaDB — neu co san thi populate, khong thi hien thi fallback
try:
    import chromadb
    HAS_CHROMA = True
    print("ChromaDB da san sang!")
except ImportError:
    HAS_CHROMA = False
    print("ChromaDB chua cai dat.")
    print("Cai dat: pip install chromadb")
    print()
    print("Fallback: Chunks da duoc tach va san sang de su dung.")
    print("Khi ChromaDB duoc cai dat, ban co the:")
    print("  1. Chay chunker.py --chroma de populate ChromaDB")
    print("  2. Su dung retriever.py de truy xuat vector search")

if HAS_CHROMA:
    # Tao ChromaDB collection va them chunks
    client = chromadb.Client()
    collection = client.get_or_create_collection(
        name="hr_policies_demo",
        metadata={"description": "HR Policy chunks - Checkpoint C1 demo"},
    )
    
    # Chuan bi du lieu
    texts = [c["content"] for c in all_chunks_all_docs]
    ids = [f"chunk-{i}" for i in range(len(all_chunks_all_docs))]
    metas = [{"doc_id": c.get("doc_id", "?"), "section": c.get("section", "?")} for c in all_chunks_all_docs]
    
    collection.add(documents=texts, ids=ids, metadatas=metas)
    
    print(f"\nChromaDB Collection: {collection.name}")
    print(f"  So chunks: {collection.count()}")
    print(f"  Metadata fields: doc_id, section")
    print("\nCollection san sang cho vector search!")

In [ ]:
# Tong ket cuoi cung
total_chunks = len(all_chunks_all_docs)
all_words = [len(c["content"].split()) for c in all_chunks_all_docs]
total_words = sum(all_words)
avg_chunk_size = total_words / total_chunks if total_chunks > 0 else 0

print("=" * 60)
print("TONG KET CHECKPOINT C1")
print("=" * 60)
print(f"So tai lieu da xu ly:  {len(documents)}")
print(f"Tong so chunks tao ra: {total_chunks}")
print(f"Trung binh tu/chunk:   {avg_chunk_size:.0f}")
print(f"Tong so tu da xu ly:   {total_words}")
print(f"ChromaDB:              {'Co' if HAS_CHROMA else 'Chua cai dat (fallback)'}")
print()
print("Phan tich chi tiet:")
for doc in documents:
    chunks = chunk_markdown_by_heading(doc["content"])
    sections = set(c["section"] for c in chunks)
    print(f"  {doc['doc_id']}: {len(chunks)} chunks, {len(sections)} sections")
print()
print("Checkpoint C1 hoan thanh!")
print("Ban da hieu cach chunker.py tach tai lieu thanh chunks.")
print("Tiep tuc lab de xay dung retriever.py va evaluator.py.")